In [39]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision.datasets import CIFAR10

In [40]:
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
    
])
trainset = CIFAR10(root="./data",train=True,download=True,transform=transform)
testset = CIFAR10(root="./data",train=False,download=True,transform=transform)

In [41]:
trainset

Dataset CIFAR10
    Number of datapoints: 50000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

In [42]:
trainLoader = DataLoader(trainset,batch_size=64,shuffle=True)
testLoader  = DataLoader(testset,batch_size=64)

# Build the CNN

In [43]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(32,64,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(64,128,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2)

        )  

        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128,256),
            nn.ReLU(),
            nn.Linear(256,10)


        )
    def forward(self,x):
        x =  self.conv_layers(x)
        x = x.view(x.size(0),-1)
        x = self.fc_layers(x)

        return x

In [44]:
model = CNN()

In [45]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

# Training the CNN

In [46]:
epochs = 10

for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images,labels in trainLoader:
        optimizer.zero_grad()
        output = model.forward(images)
        loss = criterion(output,labels)

        loss.backward()
        optimizer.step()
        epoch_training_loss+=loss.item()
    print(f"epoch = {epoch+1}/{epochs} & loss = {epoch_training_loss/len(trainLoader)}")

epoch = 1/10 & loss = 1.3613765063645589
epoch = 2/10 & loss = 0.9176558884970673
epoch = 3/10 & loss = 0.7445767340071671
epoch = 4/10 & loss = 0.6246762733020441
epoch = 5/10 & loss = 0.5183078220585728
epoch = 6/10 & loss = 0.4260188655932541
epoch = 7/10 & loss = 0.3383857907055665
epoch = 8/10 & loss = 0.26478824084219726
epoch = 9/10 & loss = 0.20399890389636427
epoch = 10/10 & loss = 0.16277128746709252


# Evaluation

In [49]:
correct_labels = 0
total_labels = 0

model.eval()
with torch.no_grad():
    for images,labels in trainLoader:
        outputs = model.forward(images)
        _,predicted = torch.max(outputs,1)

        correct_labels += (predicted==labels).sum().item()
        total_labels += labels.size(0)

print(f"Accuracy = {correct_labels/total_labels * 100}")

Accuracy = 96.71600000000001
